In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import AutoModelForQuestionAnswering
from transformers import set_seed
import torch
from time import time
import evaluate
import numpy as np

In [ ]:
torch.manual_seed(42)
set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
batch_size = 32
epochs = 4
lr = 5e-5

In [ ]:
squad_dataset = load_dataset("squad")
squad_dataset['test'] =squad_dataset['validation']
del squad_dataset['validation']
train_val_split = squad_dataset['train'].train_test_split(test_size=0.1, seed=42)
squad_dataset['train'] = train_val_split['train']
squad_dataset['val'] = train_val_split['test']


In [ ]:
def add_end_idx(example):
    gold_text = example['answers']['text'][0]
    start_idx = example['answers']['answer_start'][0]
    end_idx = start_idx + len(gold_text)

    # sometimes squad answers are off by a character or two – fix this
    if example['context'][start_idx:end_idx] == gold_text:
        example['answers']['answer_end'] = [end_idx]
    elif example['context'][start_idx-1:end_idx-1] == gold_text:
        example['answers']['answer_start'] = [start_idx - 1]
        example['answers']['answer_end'] = [end_idx - 1]  # When the gold label is off by one character
    elif example['context'][start_idx-2:end_idx-2] == gold_text:
        example['answers']['answer_start'] = [start_idx - 2]
        example['answers']['answer_end'] = [end_idx - 2]  # When the gold label is off by two characters
    else:
        example['answers']['answer_start'] = [start_idx]
        example['answers']['answer_end'] = [end_idx]
    return example


In [ ]:
squad_dataset = squad_dataset.map(add_end_idx)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModelForQuestionAnswering.from_pretrained("distilbert-base-uncased")
model = model.to(device)

In [ ]:
def preprocess_function(examples):
    # Tokenize the inputs and targets
    inputs = tokenizer(
        examples["question"],
        examples["context"],
        truncation=True,
        max_length=512,
        return_offsets_mapping=True,
    )
    offset_mapping = inputs.pop("offset_mapping")
    # Get the start and end positions of the answer
    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        answers = examples["answers"][i]

        # If there's no valid answer, set CLS token index (0)
        if len(answers["answer_start"]) == 0:
            start_positions.append(0)
            end_positions.append(0)
            continue

        start_index = answers["answer_start"][0]
        end_index = answers["answer_end"][0]
    
        

        # Find the first token that contains the start of the answer
        token_start_index = None
        for j, (start, end) in enumerate(offsets):
            if start <= start_index < end :
                token_start_index = j
                break

        # Find the last token that contains the end of the answer
        token_end_index = None
        for j, (start, end) in enumerate(offsets):
            if start < end_index <= end and j>=token_start_index:
                token_end_index = j
                break
        # If start or end token is not found, set to CLS token
        if token_start_index is None or token_end_index is None:
            start_positions.append(0)
            end_positions.append(0)
        else:
            # assert token_start_index<=token_end_index, f"Token start index {token_start_index} should be less than or equal to token end index {token_end_index}"

            start_positions.append(token_start_index)
            end_positions.append(token_end_index)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions

    return inputs

In [ ]:
tokenized_datasets = squad_dataset.map(preprocess_function, batched=True)

In [ ]:
tokenized_train,tokenized_val,tokenized_test = tokenized_datasets['train'],tokenized_datasets['val'],tokenized_datasets['test']

In [ ]:
## Sanity check if the start and end positions are correct answers should be same 
tokenized_train['answers'][2]['text']
tokenizer.decode(tokenized_train['input_ids'][2][tokenized_train['start_positions'][2]:tokenized_train['end_positions'][2]+1])

In [ ]:
squad_metric = evaluate.load("squad")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    start_logits, end_logits = predictions
    start_positions, end_positions = labels

    # Convert logits to predicted positions
    pred_start = np.argmax(start_logits, axis=1)
    pred_end = np.argmax(end_logits, axis=1)

    predictions_formatted = []
    references_formatted = []

    for i in range(len(start_positions)):
        pred_span = tokenizer.decode(tokenized_train[i]["input_ids"][pred_start[i]:pred_end[i]+1])
        true_span = tokenizer.decode(tokenized_train[i]["input_ids"][start_positions[i]:end_positions[i]+1])

        # Format for SQuAD metric
        predictions_formatted.append({"id": str(i), "prediction_text": pred_span})
        references_formatted.append({"id": str(i), "answers": {"text": [true_span], "answer_start": [start_positions[i]]}})

    # Compute official SQuAD metrics
    squad_results = squad_metric.compute(predictions=predictions_formatted, references=references_formatted)
    
    return squad_results


In [ ]:
results_path = './results/batch_size_{}_epochs_{}_lr_{}'.format(batch_size, epochs, lr)
log_path = './logs/batch_size_{}_epochs_{}_lr_{}'.format(batch_size, epochs, lr)

In [ ]:
from transformers import TrainingArguments, Trainer

In [ ]:
training_args=TrainingArguments(
    output_dir=results_path,
    num_train_epochs=epochs, 
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=256,
    learning_rate=lr,
    logging_dir=log_path,
    logging_steps=100,
    eval_strategy='steps',
    save_steps=100,
    eval_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='tensorboard',
    seed=42,
    run_name='Batch size: {}, Epochs: {}, LR: {}'.format(batch_size, epochs, lr),
    lr_scheduler_type='constant',
    warmup_steps=0,
    fp16 = False
)

In [ ]:
start_time = time()

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train, 
    eval_dataset=tokenized_val, 
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
# Save the best model
'''
Keep a local copy of the best model
'''
best_model_path = "./best_model_{batch_size}_{epochs}_{lr}".format(batch_size=batch_size, epochs=epochs, lr=lr)
trainer.model.save_pretrained(best_model_path)

In [ ]:
with open ('./time.txt', 'a+') as f:
    f.write('Batch size: {}, Epochs: {}, LR: {} - Training time: {:.2f} seconds\n'.format(batch_size, epochs, lr, time()-start_time))

In [ ]:
## Note the F1 and EM scores
trainer.predict(test_dataset=tokenized_test)